# Knowledge Graph Completion with R-GCN / CompGCN

Link prediction on large-scale KGs with a **PySpark** ETL layer and **PyTorch Geometric** encoders.

**Primary dataset:** `ogbl-biokg` (largest OGB KG that is honestly trainable on a T4 with neighbor sampling).
**Large-scale Spark job:** Wikidata5M degree/k-core subgraph (`wikidata5m_subset`).
**Smoke test:** FB15k-237.

Hardware this notebook is written for:
- Kaggle **NVIDIA T4 16 GB** (assignment default)
- Local **RTX 4050 ~6 GB** (tighter batch/fanout/dim)

Set `QUICK_RUN = True` first. Full run on Kaggle with internet **on** for the first session (installs + OGB download), then you can attach the dataset and turn internet **off**.

Viva note: full-batch R-GCN on Wikidata5M or `ogbl-wikikg2` will not fit in 6–16 GB. We never claim that it does.

In [ ]:
# Cell: Kaggle-compatible installs. Skip torch itself (Kaggle GPU image already has it).
import os, sys, subprocess, shutil, platform

def pip_install(pkgs):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + pkgs
    print(' '.join(cmd))
    subprocess.check_call(cmd)

pip_install(['pyspark==3.5.1', 'pyarrow==16.1.0', 'ogb==1.3.6', 'umap-learn==0.5.6'])

try:
    import torch_geometric  # noqa: F401
    print('torch-geometric already present')
except Exception:
    pip_install(['torch-geometric==2.5.3'])

import torch
print('torch', torch.__version__, 'cuda', torch.version.cuda, 'gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

# Optional PyG C++ extensions (NeighborLoader is happier with these). Fail soft.
if torch.cuda.is_available() and torch.version.cuda:
    tv = torch.__version__.split('+')[0]
    cu = 'cu' + torch.version.cuda.replace('.', '')
    url = f'https://data.pyg.org/whl/torch-{tv}+{cu}.html'
    try:
        pip_install(['torch-scatter', 'torch-sparse', '-f', url])
    except Exception as e:
        print('torch-scatter/sparse install skipped:', e)

print('java:', shutil.which('java'), 'platform:', platform.platform())

## CONFIG

`DATASET` is the only conceptual switch you should need:
- `biokg` — main OGB benchmark (justified below)
- `wikidata5m_subset` — Spark extracts a connected, degree-filtered subgraph that fits GPU memory
- `fb15k237` — smoke test (minutes, not hours)

`MACHINE = 'auto'` picks `kaggle_t4` vs `rtx4050` vs `cpu` from the live GPU name and VRAM.

In [ ]:
import os, time, gc, json, math, random, traceback
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# ---- edit these ----
DATASET = 'biokg'          # 'biokg' | 'wikidata5m_subset' | 'fb15k237'
QUICK_RUN = True           # True: tiny debug run; False: full assignment run
MACHINE = 'auto'           # 'auto' | 'kaggle_t4' | 'rtx4050' | 'cpu'
SEED = 42
COMPGCN_OPS = ['sub', 'mult', 'corr']  # QUICK_RUN will keep only 'mult'
RUN_ABLATIONS = not QUICK_RUN
RUN_UMAP = not QUICK_RUN

IS_KAGGLE = os.path.exists('/kaggle/working') or os.path.exists('/kaggle/input')
WORK = Path('/kaggle/working') if IS_KAGGLE else Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if not IS_KAGGLE and (WORK / 'notebooks').exists():
    ROOT = WORK
else:
    ROOT = WORK
KAGGLE_INPUT = Path('/kaggle/input') if Path('/kaggle/input').exists() else ROOT / 'data'
OUT = Path('/kaggle/working/outputs') if IS_KAGGLE else ROOT / 'outputs'
FIG = Path('/kaggle/working/figures') if IS_KAGGLE else ROOT / 'report' / 'figures'
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)
SPARK_TMP = str(Path('/kaggle/working/spark-tmp') if IS_KAGGLE else ROOT / 'spark-tmp')
Path(SPARK_TMP).mkdir(parents=True, exist_ok=True)

def detect_machine():
    if not torch.cuda.is_available():
        return 'cpu'
    name = torch.cuda.get_device_name(0).lower()
    mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f'Detected GPU: {torch.cuda.get_device_name(0)}  VRAM={mem_gb:.1f} GB')
    if '4050' in name or mem_gb < 8.5:
        return 'rtx4050'
    if 't4' in name or mem_gb >= 14:
        return 'kaggle_t4'
    return 'rtx4050' if mem_gb < 12 else 'kaggle_t4'

if MACHINE == 'auto':
    MACHINE = detect_machine()
print('MACHINE =', MACHINE, '| DATASET =', DATASET, '| QUICK_RUN =', QUICK_RUN, '| IS_KAGGLE =', IS_KAGGLE)

# T4 16 GB can take larger batches. Laptop RTX 4050 is usually 6 GB — stay conservative.
PRESETS = {
    'kaggle_t4': dict(BATCH_SIZE=1024, FANOUT=[15, 10], EMBED_DIM=64, NUM_NEG=32,
                      SPARK_DRIVER_MEMORY='12g', SHUFFLE_PARTITIONS=8, EVAL_CHUNK=4096, NUM_LAYERS=2, NUM_BASES=8),
    'rtx4050':   dict(BATCH_SIZE=256,  FANOUT=[8, 8],   EMBED_DIM=32, NUM_NEG=16,
                      SPARK_DRIVER_MEMORY='8g',  SHUFFLE_PARTITIONS=4, EVAL_CHUNK=1024, NUM_LAYERS=2, NUM_BASES=4),
    'cpu':       dict(BATCH_SIZE=128,  FANOUT=[5, 5],   EMBED_DIM=32, NUM_NEG=8,
                      SPARK_DRIVER_MEMORY='8g',  SHUFFLE_PARTITIONS=4, EVAL_CHUNK=512,  NUM_LAYERS=1, NUM_BASES=4),
}
P = PRESETS[MACHINE].copy()
if QUICK_RUN:
    P.update(dict(BATCH_SIZE=min(P['BATCH_SIZE'], 128), FANOUT=[5], EMBED_DIM=32, NUM_NEG=4, NUM_LAYERS=1, NUM_BASES=4))
    COMPGCN_OPS = ['mult']
    EPOCHS = 2
    MAX_TRAIN_TRIPLES = 20000
    MAX_EVAL_TRIPLES = 1000
else:
    EPOCHS = 8 if MACHINE == 'rtx4050' else 15
    MAX_TRAIN_TRIPLES = None
    MAX_EVAL_TRIPLES = 20000 if MACHINE == 'rtx4050' else None

LR = 1e-3
WEIGHT_DECAY = 1e-5
GRAD_CLIP = 1.0
DROPOUT = 0.2
PATIENCE = 4
NUM_NEG = P['NUM_NEG']
BATCH_SIZE = P['BATCH_SIZE']
FANOUT = P['FANOUT']
EMBED_DIM = P['EMBED_DIM']
NUM_LAYERS = P['NUM_LAYERS']
NUM_BASES = P['NUM_BASES']
EVAL_CHUNK = P['EVAL_CHUNK']
MIN_DEGREE = 5
TOP_K_ENTITIES = 40000 if MACHINE == 'rtx4050' else 80000
K_CORE = 2
ADD_INVERSE = True
N_CANDIDATES = 200 if MACHINE == 'rtx4050' else 500  # filtered ranking candidates for non-OGB graphs
OOM_FALLBACK = dict(BATCH_SIZE=64, FANOUT=[4], EMBED_DIM=16, NUM_NEG=4, NUM_LAYERS=1)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

def mem_report(tag=''):
    extra = ''
    if torch.cuda.is_available():
        extra = f' | cuda_alloc={torch.cuda.memory_allocated()/1e9:.2f}GB peak={torch.cuda.max_memory_allocated()/1e9:.2f}GB'
    print(f'[mem {tag}] ram_check_ok{extra}')

print('preset', P)
print('DEVICE', DEVICE, 'AMP', USE_AMP)
mem_report('start')

## Why these datasets?

| Dataset | Entities | Relations | Train triples | T4 full-batch GNN | T4 / 4050 mini-batch | Availability |
|---|---:|---:|---:|---|---|---|
| ogbl-biokg | 93,773 | 51 | 4.76M | tight / no | **yes** | OGB downloader or `/kaggle/input` |
| ogbl-wikikg2 | 2.50M | 535 | 16.1M | **no** | entity table already hundreds of MB; sampling over 2.5M nodes is RAM-heavy | OGB |
| Wikidata5M | 4.59M | 822 | 20.6M | **no** | embeddings alone ~1 GB at dim 64 | KEPLER dump / Kaggle dataset |
| FB15k-237 | 14,541 | 237 | 272k | yes | yes | small files / GitHub |
| WN18RR | 40,943 | 11 | 87k | yes | yes | small files |

**Decision:** `ogbl-biokg` is the largest *official OGB* graph we can train with neighbor sampling on a T4 without lying about memory. Wikidata5M is processed in Spark and reduced to a degree/k-core subgraph (`TOP_K_ENTITIES`) so the "large-scale KG" claim is about **data processing**, not full-batch GNN on 4.8M nodes. FB15k-237 is the sanity check.

Why **PyTorch Geometric** rather than DGL: OGB's Python API and `RGCNConv` / `NeighborLoader` examples are PyG-first; matching DGL wheels to the Kaggle CUDA build is more brittle. CompGCN is implemented from the paper (not a black-box DGL class) so the viva can walk through the composition operators.

In [ ]:
CANDIDATES = [
    dict(name='ogbl-biokg', entities=93773, relations=51, triples=4762678, mem='~0.2 GB raw', t4_full='no/tight', t4_mini='yes', avail='OGB / Kaggle'),
    dict(name='ogbl-wikikg2', entities=2500604, relations=535, triples=16109182, mem='~1-2 GB', t4_full='no', t4_mini='very hard', avail='OGB'),
    dict(name='Wikidata5M', entities=4594485, relations=822, triples=20614279, mem='~1-2 GB text', t4_full='no', t4_mini='no (use subset)', avail='KEPLER / Kaggle'),
    dict(name='FB15k-237', entities=14541, relations=237, triples=272115, mem='<20 MB', t4_full='yes', t4_mini='yes', avail='mirrors'),
    dict(name='WN18RR', entities=40943, relations=11, triples=86835, mem='<10 MB', t4_full='yes', t4_mini='yes', avail='mirrors'),
]
print(pd.DataFrame(CANDIDATES).to_string(index=False))
print('\nSelected DATASET =', DATASET)
print('Justification: biokg is the largest OGB KG that fits neighbor-sampled R-GCN/CompGCN on T4 16GB; 4050 uses the same pipeline with smaller dim/batch.')

## Spark session (Big Data layer)

Local mode on Kaggle: `local[*]`, Arrow on, shuffle partitions small (single node). Driver memory is **12g on T4 / 8g on 4050** so PyTorch still has RAM left. **Stop Spark before training.**

In [ ]:
from pyspark.sql import SparkSession, Window, functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

def build_spark():
    spark = (
        SparkSession.builder.master('local[*]')
        .appName('kg-completion')
        .config('spark.driver.memory', P['SPARK_DRIVER_MEMORY'])
        .config('spark.driver.maxResultSize', '4g')
        .config('spark.sql.shuffle.partitions', str(P['SHUFFLE_PARTITIONS']))
        .config('spark.sql.execution.arrow.pyspark.enabled', 'true')
        .config('spark.sql.execution.arrow.pyspark.fallback.enabled', 'true')
        .config('spark.local.dir', SPARK_TMP)
        .config('spark.ui.showConsoleProgress', 'true')
        .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer')
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel('WARN')
    return spark

spark = build_spark()
print(spark.version, spark.sparkContext.master, 'defaultParallelism', spark.sparkContext.defaultParallelism)

TRIPLE_SCHEMA = StructType([
    StructField('h', StringType(), False),
    StructField('r', StringType(), False),
    StructField('t', StringType(), False),
])

## Load triples (internet optional)

Search order: `/kaggle/input/...` (internet off) → OGB downloader (internet on) → small GitHub FB15k-237 → synthetic fallback so `QUICK_RUN` never dies.

In [ ]:
def find_dir_with(*names):
    root = KAGGLE_INPUT
    if not root.exists():
        return None
    for p in [root, *root.glob('*')]:
        if all((p / n).exists() for n in names):
            return p
        for q in p.rglob('train.txt'):
            if all((q.parent / n).exists() for n in names):
                return q.parent
    return None

def read_triple_file(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 3:
                rows.append((parts[0], parts[1], parts[2]))
    return rows

def synthetic_kg(n_ent=500, n_rel=10, n_train=4000, n_valid=400, n_test=400):
    print('WARNING: using synthetic KG (no dataset found). Fine for pipeline debug, not for the report tables.')
    rng = np.random.default_rng(SEED)
    def samp(n):
        h = rng.integers(0, n_ent, size=n)
        r = rng.integers(0, n_rel, size=n)
        t = (h + r + rng.integers(1, 6, size=n)) % n_ent
        return list(zip(h.astype(str), r.astype(str), t.astype(str)))
    return {'train': samp(n_train), 'valid': samp(n_valid), 'test': samp(n_test), 'source': 'synthetic'}

def load_fb15k237():
    d = find_dir_with('train.txt', 'valid.txt', 'test.txt')
    if d is None:
        # common Kaggle dataset folder names
        for cand in KAGGLE_INPUT.glob('**/*') if KAGGLE_INPUT.exists() else []:
            if cand.name.lower().startswith('fb15'):
                d = cand if cand.is_dir() else cand.parent
                break
    if d is not None and (d / 'train.txt').exists():
        return {s: read_triple_file(d / f'{s}.txt') for s in ['train', 'valid', 'test']} | {'source': str(d)}
    try:
        import urllib.request
        base = 'https://raw.githubusercontent.com/ZhenfengLei/KGDatasets/master/FB15k-237'
        dest = OUT / 'fb15k237'; dest.mkdir(exist_ok=True)
        splits = {}
        for s in ['train', 'valid', 'test']:
            fp = dest / f'{s}.txt'
            if not fp.exists():
                urllib.request.urlretrieve(f'{base}/{s}.txt', fp)
            splits[s] = read_triple_file(fp)
        splits['source'] = str(dest)
        return splits
    except Exception as e:
        print('FB15k-237 download failed:', e)
        return synthetic_kg()

def load_wikidata5m():
    d = find_dir_with('wikidata5m_transductive_train.txt') 
    if d is None:
        d = find_dir_with('train.txt')
    aliases = {
        'train': ['wikidata5m_transductive_train.txt', 'train.txt'],
        'valid': ['wikidata5m_transductive_valid.txt', 'valid.txt'],
        'test': ['wikidata5m_transductive_test.txt', 'test.txt'],
    }
    if d is None:
        print('Wikidata5M not found under', KAGGLE_INPUT, '- attach a Kaggle dataset or switch DATASET.')
        return synthetic_kg(n_ent=2000, n_rel=20, n_train=8000, n_valid=500, n_test=500)
    out = {'source': str(d)}
    for split, names in aliases.items():
        path = next((d / n for n in names if (d / n).exists()), None)
        out[split] = read_triple_file(path)
    return out

def load_biokg():
    # Prefer a pre-attached OGB cache so internet can be off.
    ogb_roots = []
    if KAGGLE_INPUT.exists():
        ogb_roots += [p for p in KAGGLE_INPUT.glob('**') if p.is_dir() and 'ogbl_biokg' in p.name.replace('-', '_') or (p / 'ogbl_biokg').exists() or 'biokg' in p.name.lower()]
    ogb_roots.append(OUT / 'ogb')
    last_err = None
    from ogb.linkproppred import LinkPropPredDataset
    for root in ogb_roots:
        try:
            ds = LinkPropPredDataset(name='ogbl-biokg', root=str(root))
            split = ds.get_edge_split()
            graph = ds[0]
            def pack(part):
                h, r, t = part['head'], part['relation'], part['tail']
                rows = list(zip(h.astype(str), r.astype(str), t.astype(str)))
                neg = {}
                if 'head_neg' in part:
                    neg['head_neg'] = np.asarray(part['head_neg'])
                    neg['tail_neg'] = np.asarray(part['tail_neg'])
                return rows, neg
            train, _ = pack(split['train'])
            valid, vneg = pack(split['valid'])
            test, tneg = pack(split['test'])
            return {
                'train': train, 'valid': valid, 'test': test,
                'valid_neg': vneg, 'test_neg': tneg,
                'source': f'ogbl-biokg:{root}',
                'n_ent_hint': int(graph['num_nodes']) if isinstance(graph, dict) else None,
            }
        except Exception as e:
            last_err = e
            continue
    print('ogbl-biokg load failed:', last_err)
    print('Attach the OGB files under /kaggle/input or enable internet.')
    return synthetic_kg(n_ent=2000, n_rel=20, n_train=8000)

LOADERS = {'biokg': load_biokg, 'fb15k237': load_fb15k237, 'wikidata5m_subset': load_wikidata5m}
raw = LOADERS[DATASET]()
print('source', raw['source'])
print({k: len(raw[k]) for k in ['train', 'valid', 'test']})
if QUICK_RUN:
    for k in ['train', 'valid', 'test']:
        cap = MAX_TRAIN_TRIPLES if k == 'train' else MAX_EVAL_TRIPLES
        raw[k] = raw[k][:cap]
    print('truncated for QUICK_RUN', {k: len(raw[k]) for k in ['train', 'valid', 'test']})

## PySpark ETL

Cleaning → statistics → relation categories (1-1 / 1-N / N-1 / N-N) → optional degree + k-core subgraph → dense ids → filter unseen-entity test triples → inverse relations → filtered-eval lookup tables → Parquet.

In [ ]:
def rows_to_sdf(rows):
    return spark.createDataFrame(rows, schema=TRIPLE_SCHEMA)

t0 = time.time()
train_sdf = rows_to_sdf(raw['train'])
valid_sdf = rows_to_sdf(raw['valid'])
test_sdf = rows_to_sdf(raw['test'])
print('spark ingest s', round(time.time() - t0, 2))

def clean_triples(df, drop_self_loops=True):
    out = df.dropna().dropDuplicates(['h', 'r', 't'])
    out = out.filter((F.length('h') > 0) & (F.length('r') > 0) & (F.length('t') > 0))
    if drop_self_loops:
        out = out.filter(F.col('h') != F.col('t'))
    return out

n_raw_train = train_sdf.count()
train_sdf = clean_triples(train_sdf)
valid_sdf = clean_triples(valid_sdf)
test_sdf = clean_triples(test_sdf)
print('train triples raw/clean', n_raw_train, train_sdf.count())

def entity_degree(df):
    return (df.select(F.col('h').alias('e')).union(df.select(F.col('t').alias('e')))
            .groupBy('e').count().withColumnRenamed('count', 'degree'))

def relation_categories(df, thr=1.5):
    tph = df.groupBy('r', 'h').agg(F.countDistinct('t').alias('nt')).groupBy('r').agg(F.avg('nt').alias('tph'))
    hpt = df.groupBy('r', 't').agg(F.countDistinct('h').alias('nh')).groupBy('r').agg(F.avg('nh').alias('hpt'))
    j = tph.join(hpt, 'r')
    return j.withColumn(
        'category',
        F.when((F.col('tph') < thr) & (F.col('hpt') < thr), '1-1')
         .when((F.col('tph') >= thr) & (F.col('hpt') < thr), '1-N')
         .when((F.col('tph') < thr) & (F.col('hpt') >= thr), 'N-1')
         .otherwise('N-N')
    )

deg = entity_degree(train_sdf).cache()
relfreq = train_sdf.groupBy('r').count().withColumnRenamed('count', 'freq')
relcat = relation_categories(train_sdf).cache()

n_ent = train_sdf.select('h').union(train_sdf.select('t')).distinct().count()
n_rel = train_sdf.select('r').distinct().count()
n_tri = train_sdf.count()
deg_pd = deg.select('degree').toPandas()['degree']
cat_pd = relcat.groupBy('category').count().toPandas()
print('entities', n_ent, 'relations', n_rel, 'triples', n_tri)
print('degree min/median/mean/p99/max', int(deg_pd.min()), int(deg_pd.median()), float(deg_pd.mean()), float(deg_pd.quantile(0.99)), int(deg_pd.max()))
print(cat_pd.to_string(index=False))

spark_stats = {
    'n_entities': int(n_ent), 'n_relations': int(n_rel), 'n_triples': int(n_tri),
    'degree_min': int(deg_pd.min()), 'degree_median': float(deg_pd.median()),
    'degree_mean': float(deg_pd.mean()), 'degree_p99': float(deg_pd.quantile(0.99)),
    'degree_max': int(deg_pd.max()),
    'rel_categories': {str(r['category']): int(r['count']) for _, r in cat_pd.iterrows()},
    'duplicates_or_loops_removed': int(n_raw_train - n_tri),
}

In [ ]:
def degree_filter(df, min_degree, top_k):
    d = entity_degree(df).filter(F.col('degree') >= min_degree).orderBy(F.desc('degree')).limit(int(top_k))
    keep = d.select(F.col('e').alias('id'))
    return (df.join(keep.withColumnRenamed('id', 'h'), 'h')
              .join(keep.withColumnRenamed('id', 't'), 't')
              .select('h', 'r', 't'))

def k_core(df, k, max_iters=12):
    cur = df
    for i in range(max_iters):
        d = entity_degree(cur).filter(F.col('degree') >= k)
        nxt = (cur.join(d.select(F.col('e').alias('h')), 'h')
                  .join(d.select(F.col('e').alias('t')), 't')
                  .select('h', 'r', 't'))
        n0, n1 = cur.count(), nxt.count()
        print(f' k-core iter {i}: {n0} -> {n1}')
        if n1 == n0:
            return nxt
        cur = nxt
    return cur

subset_stats = {}
if DATASET == 'wikidata5m_subset':
    print('Extracting degree/k-core subgraph so GPU training can actually run...')
    before = {'entities': n_ent, 'triples': n_tri}
    train_sdf = degree_filter(train_sdf, MIN_DEGREE, TOP_K_ENTITIES)
    train_sdf = k_core(train_sdf, K_CORE)
    keep_e = train_sdf.select(F.col('h').alias('e')).union(train_sdf.select(F.col('t').alias('e'))).distinct()
    keep_r = train_sdf.select('r').distinct()
    valid_sdf = valid_sdf.join(keep_e.withColumnRenamed('e', 'h'), 'h').join(keep_e.withColumnRenamed('e', 't'), 't').join(keep_r, 'r').select('h', 'r', 't')
    test_sdf = test_sdf.join(keep_e.withColumnRenamed('e', 'h'), 'h').join(keep_e.withColumnRenamed('e', 't'), 't').join(keep_r, 'r').select('h', 'r', 't')
    n_ent = keep_e.count(); n_rel = keep_r.count(); n_tri = train_sdf.count()
    subset_stats = {'before': before, 'after': {'entities': n_ent, 'relations': n_rel, 'triples': n_tri}}
    print('subset', subset_stats)

# dense 0-based ids (window, not StringIndexer, so ids are stable and dense)
ents = train_sdf.select(F.col('h').alias('name')).union(train_sdf.select(F.col('t').alias('name'))).distinct()
rels = train_sdf.select(F.col('r').alias('name')).distinct()
e_map = ents.withColumn('id', F.row_number().over(Window.orderBy('name')) - 1)
r_map = rels.withColumn('id', F.row_number().over(Window.orderBy('name')) - 1)

def index_df(df):
    return (df.join(e_map.withColumnRenamed('name', 'h').withColumnRenamed('id', 'h_id'), 'h')
              .join(r_map.withColumnRenamed('name', 'r').withColumnRenamed('id', 'r_id'), 'r')
              .join(e_map.withColumnRenamed('name', 't').withColumnRenamed('id', 't_id'), 't')
              .select(F.col('h_id').alias('h'), F.col('r_id').alias('r'), F.col('t_id').alias('t')))

train_id = index_df(train_sdf)
valid_id = index_df(valid_sdf)
test_id = index_df(test_sdf)

# drop valid/test triples whose entities never appear in train (indexed inner join already does this)
print('indexed train/valid/test', train_id.count(), valid_id.count(), test_id.count())

n_rel_id = r_map.count()
n_ent_id = e_map.count()
if ADD_INVERSE:
    inv = train_id.select(F.col('t').alias('h'), (F.col('r') + F.lit(n_rel_id)).alias('r'), F.col('h').alias('t'))
    train_msg = train_id.unionByName(inv)
    n_rel_msg = 2 * n_rel_id
else:
    train_msg = train_id
    n_rel_msg = n_rel_id
print('message-passing rels', n_rel_msg, 'entities', n_ent_id)

all_true = train_id.unionByName(valid_id).unionByName(test_id)
true_tails = all_true.groupBy('h', 'r').agg(F.collect_set('t').alias('true_set'))
true_heads = all_true.groupBy('t', 'r').agg(F.collect_set('h').alias('true_set'))

parquet_dir = OUT / 'parquet'
for name, df in [('train', train_id), ('valid', valid_id), ('test', test_id),
                 ('train_msg', train_msg), ('e_map', e_map), ('r_map', r_map),
                 ('true_tails', true_tails), ('true_heads', true_heads), ('relcat', relcat)]:
    path = str(parquet_dir / name)
    df.coalesce(4).write.mode('overwrite').parquet(path)
    print('wrote', path)

relcat_pd = relcat.select('r', 'category', 'tph', 'hpt').toPandas()
spark_stats.update({'n_entities_indexed': int(n_ent_id), 'n_relations_indexed': int(n_rel_id),
                    'n_rel_msg': int(n_rel_msg), 'subset': subset_stats})
print('ETL wall-clock s', round(time.time() - t0, 2))

## Spark scalability experiment

Time a representative job (count + degree + relation frequency) at 10/25/50/100% of training triples. This is the figure the report needs for the Big Data claim.

In [ ]:
fracs = [0.10, 0.25, 0.50, 1.00] if not QUICK_RUN else [0.25, 1.00]
scale_rows = []
full_n = train_sdf.count()
for fr in fracs:
    part = train_sdf.sample(False, fr, seed=SEED) if fr < 1 else train_sdf
    spark.sparkContext._jvm.System.gc()
    t1 = time.time()
    _ = part.count()
    _ = entity_degree(part).agg(F.avg('degree')).collect()
    _ = part.groupBy('r').count().count()
    dt = time.time() - t1
    n = int(full_n * fr) if fr < 1 else full_n
    scale_rows.append({'fraction': fr, 'approx_triples': n, 'seconds': dt})
    print(scale_rows[-1])

scale_df = pd.DataFrame(scale_rows)
plt.figure(figsize=(5.2, 3.4))
plt.plot(scale_df['fraction'] * 100, scale_df['seconds'], marker='o')
plt.xlabel('% of training triples')
plt.ylabel('Spark job time (s)')
plt.title('Spark ETL scaling (count + degree + rel freq)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG / 'fig_spark_scale.png', dpi=160)
plt.show()
scale_df.to_csv(OUT / 'spark_scale.csv', index=False)

In [ ]:
plt.figure(figsize=(5.2, 3.4))
vals, bins = np.histogram(deg_pd, bins=50)
plt.loglog((bins[:-1] + bins[1:]) / 2 + 1e-9, np.maximum(vals, 1), marker='.', linestyle='none')
plt.xlabel('degree')
plt.ylabel('count')
plt.title('Entity degree distribution (log-log)')
plt.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG / 'fig_degree_loglog.png', dpi=160)
plt.show()

# Convert Spark → numpy, then STOP Spark so GPU training has RAM.
train_np = np.array(train_id.select('h', 'r', 't').toPandas()[['h', 'r', 't']].to_numpy(), dtype=np.int64)
valid_np = np.array(valid_id.select('h', 'r', 't').toPandas()[['h', 'r', 't']].to_numpy(), dtype=np.int64)
test_np = np.array(test_id.select('h', 'r', 't').toPandas()[['h', 'r', 't']].to_numpy(), dtype=np.int64)
msg_np = np.array(train_msg.select('h', 'r', 't').toPandas()[['h', 'r', 't']].to_numpy(), dtype=np.int64)
relcat_map = {}
name_to_rid = {row['name']: int(row['id']) for row in r_map.select('name', 'id').toPandas().to_dict('records')}
for _, row in relcat_pd.iterrows():
    if row['r'] in name_to_rid:
        relcat_map[name_to_rid[row['r']]] = row['category']

tt = true_tails.toPandas()
true_tail_dict = {(int(r.h), int(r.r)): set(map(int, r.true_set)) for r in tt.itertuples()}

N_ENT, N_REL, N_REL_MSG = int(n_ent_id), int(n_rel_id), int(n_rel_msg)
print('tensors', train_np.shape, valid_np.shape, test_np.shape, 'N_ENT', N_ENT, 'N_REL', N_REL)

spark.stop()
spark = None
gc.collect()
print('Spark stopped')
mem_report('after-spark')

## Models

- **DistMult** (Yang et al., 2015): $f(h,r,t)=\langle \mathbf{h},\mathbf{r},\mathbf{t}\rangle$. No message passing — the baseline that tells you whether the GNN is doing anything.
- **R-GCN** (Schlichtkrull et al., 2018): relation-specific weights with **basis decomposition** (`num_bases`) so 51–800 relations do not explode the parameter count. Neighbor sampling, not full-batch.
- **CompGCN** (Vashishth et al., 2020): compose entity and relation vectors with $\phi\in\{\text{sub},\text{mult},\text{corr}\}$ then aggregate. Inverse relations share a direction-specific matrix.

Decoder for all GNN runs: DistMult (required). Optional TransE scoring is a one-line switch.

In [ ]:
def ccorr(a, b):
    n = a.size(-1)
    return torch.fft.irfft(torch.conj(torch.fft.rfft(a, dim=-1)) * torch.fft.rfft(b, dim=-1), n=n, dim=-1)

class DistMultNet(nn.Module):
    def __init__(self, n_ent, n_rel, dim):
        super().__init__()
        self.ent = nn.Embedding(n_ent, dim)
        self.rel = nn.Embedding(n_rel, dim)
        nn.init.xavier_uniform_(self.ent.weight)
        nn.init.xavier_uniform_(self.rel.weight)
    def score_ids(self, h, r, t):
        return (self.ent(h) * self.rel(r) * self.ent(t)).sum(-1)
    def score_emb(self, he, r, te):
        return (he * self.rel(r) * te).sum(-1)

class RGCNNet(nn.Module):
    def __init__(self, n_ent, n_rel_msg, n_rel, dim, n_layers, n_bases, dropout):
        super().__init__()
        from torch_geometric.nn import RGCNConv
        self.ent = nn.Embedding(n_ent, dim)
        self.rel = nn.Embedding(n_rel, dim)
        self.convs = nn.ModuleList([RGCNConv(dim, dim, num_relations=n_rel_msg, num_bases=n_bases) for _ in range(n_layers)])
        self.dropout = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.ent.weight)
        nn.init.xavier_uniform_(self.rel.weight)
    def encode(self, n_id, edge_index, edge_type):
        h = self.ent(n_id)
        for i, conv in enumerate(self.convs):
            h = conv(h, edge_index, edge_type)
            if i < len(self.convs) - 1:
                h = self.dropout(F.relu(h))
        return h
    def score_emb(self, he, r, te):
        return (he * self.rel(r) * te).sum(-1)

class CompGCNLayer(nn.Module):
    def __init__(self, dim, op):
        super().__init__()
        self.op = op
        self.w_in = nn.Linear(dim, dim, bias=False)
        self.w_out = nn.Linear(dim, dim, bias=False)
        self.w_loop = nn.Linear(dim, dim, bias=False)
        self.w_rel = nn.Linear(dim, dim, bias=False)
    def compose(self, e, r):
        if self.op == 'sub':
            return e - r
        if self.op == 'mult':
            return e * r
        return ccorr(e, r)
    def forward(self, x, rel, edge_index, edge_type):
        src, dst = edge_index
        n_orig = max(rel.size(0) // 2, 1)
        rel_id = torch.remainder(edge_type, n_orig)
        inv = edge_type >= n_orig
        composed = self.compose(x[src], rel[rel_id])
        msg = torch.where(inv.unsqueeze(-1), self.w_out(composed), self.w_in(composed))
        agg = torch.zeros_like(x)
        agg.index_add_(0, dst, msg)
        deg = torch.zeros(x.size(0), device=x.device, dtype=x.dtype)
        deg.index_add_(0, dst, torch.ones_like(dst, dtype=x.dtype))
        h = self.w_loop(x) + agg / deg.clamp(min=1).unsqueeze(-1)
        return h, self.w_rel(rel)

class CompGCNNet(nn.Module):
    def __init__(self, n_ent, n_rel, dim, n_layers, op, dropout):
        super().__init__()
        self.ent = nn.Embedding(n_ent, dim)
        self.rel = nn.Embedding(n_rel, dim)
        self.layers = nn.ModuleList([CompGCNLayer(dim, op) for _ in range(n_layers)])
        self.dropout = nn.Dropout(dropout)
        self.op = op
        nn.init.xavier_uniform_(self.ent.weight)
        nn.init.xavier_uniform_(self.rel.weight)
    def encode(self, n_id, edge_index, edge_type):
        h = self.ent(n_id); r = self.rel.weight
        for i, layer in enumerate(self.layers):
            h, r = layer(h, r, edge_index, edge_type)
            if i < len(self.layers) - 1:
                h = self.dropout(F.relu(h))
        self._rel_out = r
        return h
    def score_emb(self, he, r_id, te):
        r = self.rel(r_id)
        return (he * r * te).sum(-1)

print('model classes ready')

## Neighbor-sampled training

We build a CSR of the (possibly inverse-augmented) graph **once**, then each mini-batch:
1. takes `BATCH_SIZE` triples,
2. uses heads+tails as seeds,
3. samples `FANOUT` neighbors per hop,
4. runs the encoder on that subgraph,
5. DistMult-scores the true tails vs `NUM_NEG` random tails (BCE with logits),
6. AMP + grad clip.

If CUDA OOM: the loop halves `BATCH_SIZE` and `FANOUT` automatically (see `OOM_FALLBACK`).

In [ ]:
def build_csr(triples_msg, n_ent):
    src = torch.as_tensor(triples_msg[:, 0], dtype=torch.long)
    dst = torch.as_tensor(triples_msg[:, 2], dtype=torch.long)
    et = torch.as_tensor(triples_msg[:, 1], dtype=torch.long)
    order = torch.argsort(dst)
    dst, src, et = dst[order], src[order], et[order]
    cnt = torch.bincount(dst, minlength=n_ent)
    ptr = torch.zeros(n_ent + 1, dtype=torch.long)
    ptr[1:] = torch.cumsum(cnt, 0)
    return ptr, src, et

PTR, NBR, ET = build_csr(msg_np, N_ENT)
print('CSR nnz', NBR.numel(), 'mean deg', float(NBR.numel() / max(N_ENT, 1)))

def sample_subgraph(seeds, ptr, nbr, et, fanouts):
    """Fanout neighbor sample. Returns local edge_index, edge_type, n_id, mapping dict-like tensor."""
    node_list = [seeds.cpu()]
    seen = set(seeds.cpu().tolist())
    e_s, e_d, e_r = [], [], []
    current = seeds.cpu()
    for fo in fanouts:
        nxt = []
        for u in current.tolist():
            a, b = int(ptr[u]), int(ptr[u + 1])
            if a == b:
                continue
            idx = np.arange(a, b)
            if len(idx) > fo:
                idx = np.random.choice(idx, size=fo, replace=False)
            for j in idx:
                v = int(nbr[j])
                e_s.append(v); e_d.append(u); e_r.append(int(et[j]))
                if v not in seen:
                    seen.add(v); nxt.append(v)
        current = torch.tensor(nxt, dtype=torch.long) if nxt else torch.zeros(0, dtype=torch.long)
        if current.numel() == 0:
            break
        node_list.append(current)
    n_id = torch.tensor(sorted(seen), dtype=torch.long)
    g2l = {int(g): i for i, g in enumerate(n_id.tolist())}
    if e_s:
        ei = torch.tensor([[g2l[s] for s in e_s], [g2l[d] for d in e_d]], dtype=torch.long)
        er = torch.tensor(e_r, dtype=torch.long)
    else:
        ei = torch.zeros(2, 0, dtype=torch.long)
        er = torch.zeros(0, dtype=torch.long)
    mapper = torch.full((int(n_id.max()) + 1 if n_id.numel() else 1,), -1, dtype=torch.long)
    # mapper keyed by global id needs size N_ENT
    mapper = torch.full((N_ENT,), -1, dtype=torch.long)
    mapper[n_id] = torch.arange(n_id.numel())
    return ei, er, n_id, mapper

def bce(pos, neg):
    return F.binary_cross_entropy_with_logits(pos, torch.ones_like(pos)) + F.binary_cross_entropy_with_logits(neg, torch.zeros_like(neg))

def iter_triple_batches(arr, bs, shuffle=True):
    idx = np.random.permutation(len(arr)) if shuffle else np.arange(len(arr))
    for i in range(0, len(arr), bs):
        yield torch.as_tensor(arr[idx[i:i+bs]], dtype=torch.long)

from torch.cuda.amp import GradScaler, autocast

def train_model(model, name, epochs=EPOCHS, batch_size=BATCH_SIZE, fanout=FANOUT, num_neg=NUM_NEG):
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = GradScaler(enabled=USE_AMP)
    hist = {'loss': [], 'val_mrr': [], 'epoch_s': []}
    best, bad, best_state = -1.0, 0, None
    use_gnn = hasattr(model, 'encode')
    print(f'==== train {name} epochs={epochs} bs={batch_size} fanout={fanout} neg={num_neg} ====')
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); running = 0.0; steps = 0
        try:
            for batch in iter_triple_batches(train_np, batch_size):
                h, r, t = batch[:, 0], batch[:, 1], batch[:, 2]
                neg_t = torch.randint(0, N_ENT, (h.size(0), num_neg))
                opt.zero_grad(set_to_none=True)
                with autocast(enabled=USE_AMP):
                    if use_gnn:
                        seeds = torch.unique(torch.cat([h, t], 0))
                        ei, er, n_id, mapper = sample_subgraph(seeds, PTR, NBR, ET, fanout)
                        z = model.encode(n_id.to(DEVICE), ei.to(DEVICE), er.to(DEVICE))
                        m = mapper.to(DEVICE)
                        he, te = z[m[h.to(DEVICE)]], z[m[t.to(DEVICE)]]
                        # negatives may sit outside the subgraph → fall back to embedding table
                        te_neg = model.ent(neg_t.to(DEVICE))
                        pos = model.score_emb(he, r.to(DEVICE), te)
                        neg = model.score_emb(he.unsqueeze(1).expand(-1, num_neg, -1).reshape(-1, he.size(-1)),
                                              r.to(DEVICE).repeat_interleave(num_neg),
                                              te_neg.reshape(-1, he.size(-1))).view(h.size(0), num_neg)
                    else:
                        pos = model.score_ids(h.to(DEVICE), r.to(DEVICE), t.to(DEVICE))
                        neg = model.score_ids(h.to(DEVICE).unsqueeze(1).expand(-1, num_neg).reshape(-1),
                                              r.to(DEVICE).unsqueeze(1).expand(-1, num_neg).reshape(-1),
                                              neg_t.to(DEVICE).reshape(-1)).view(h.size(0), num_neg)
                    loss = bce(pos, neg)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(opt); scaler.update()
                running += float(loss.detach()); steps += 1
        except torch.cuda.OutOfMemoryError:
            print('OOM — applying fallback', OOM_FALLBACK)
            torch.cuda.empty_cache()
            return train_model(model, name, epochs=max(epochs, 1),
                               batch_size=OOM_FALLBACK['BATCH_SIZE'],
                               fanout=OOM_FALLBACK['FANOUT'],
                               num_neg=OOM_FALLBACK['NUM_NEG'])
        dt = time.time() - t0
        avg = running / max(steps, 1)
        model.eval()
        val = eval_mrr(model, valid_np[:MAX_EVAL_TRIPLES] if MAX_EVAL_TRIPLES else valid_np, use_gnn, fanout)
        hist['loss'].append(avg); hist['val_mrr'].append(val); hist['epoch_s'].append(dt)
        print(f'{name} ep {ep:02d} loss={avg:.4f} val_MRR={val:.4f} time={dt:.1f}s', end=' ')
        mem_report(f'{name}-ep{ep}')
        if val > best:
            best, bad = val, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            torch.save(best_state, OUT / f'{name}.pt')
        else:
            bad += 1
            if bad >= PATIENCE:
                print('early stop'); break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, hist

@torch.no_grad()
def eval_mrr(model, triples, use_gnn, fanout, candidates=N_CANDIDATES):
    if len(triples) == 0:
        return 0.0
    model.eval()
    ranks = []
    bs = 64 if MACHINE == 'rtx4050' else 128
    for i in range(0, len(triples), bs):
        batch = torch.as_tensor(triples[i:i+bs], dtype=torch.long)
        h, r, t = batch[:, 0], batch[:, 1], batch[:, 2]
        B = h.size(0)
        # candidate set: random entities + true tail (OGB path handled separately below)
        neg = torch.randint(0, N_ENT, (B, candidates))
        all_t = torch.cat([t.unsqueeze(1), neg], 1)  # [B, 1+C]
        if use_gnn:
            seeds = torch.unique(torch.cat([h, all_t.reshape(-1)], 0))
            # cap seeds on 4050
            if seeds.numel() > 4000 and MACHINE == 'rtx4050':
                seeds = torch.unique(torch.cat([h, t], 0))
                z_table = model.ent.weight
                he = z_table[h.to(DEVICE)]; te_all = z_table[all_t.to(DEVICE)]
                sc = model.score_emb(he.unsqueeze(1).expand(-1, all_t.size(1), -1).reshape(-1, he.size(-1)),
                                     r.to(DEVICE).repeat_interleave(all_t.size(1)),
                                     te_all.reshape(-1, he.size(-1))).view(B, -1)
            else:
                ei, er, n_id, mapper = sample_subgraph(seeds, PTR, NBR, ET, fanout)
                z = model.encode(n_id.to(DEVICE), ei.to(DEVICE), er.to(DEVICE))
                m = mapper.to(DEVICE)
                he = z[m[h.to(DEVICE)]]
                te_all = model.ent(all_t.to(DEVICE))
                sc = model.score_emb(he.unsqueeze(1).expand(-1, all_t.size(1), -1).reshape(-1, he.size(-1)),
                                     r.to(DEVICE).repeat_interleave(all_t.size(1)),
                                     te_all.reshape(-1, he.size(-1))).view(B, -1)
        else:
            he = model.ent(h.to(DEVICE))
            te_all = model.ent(all_t.to(DEVICE))
            sc = model.score_emb(he.unsqueeze(1).expand(-1, all_t.size(1), -1).reshape(-1, he.size(-1)),
                                 r.to(DEVICE).repeat_interleave(all_t.size(1)),
                                 te_all.reshape(-1, he.size(-1))).view(B, -1)
        # filtered: mask known true tails among candidates (except the current true)
        for bi in range(B):
            known = true_tail_dict.get((int(h[bi]), int(r[bi])), set())
            for j in range(1, all_t.size(1)):
                if int(all_t[bi, j]) in known:
                    sc[bi, j] = -1e9
        pos = sc[:, 0]
        rk = (sc[:, 1:] >= pos.unsqueeze(1)).sum(1) + 1
        ranks.append(rk.detach().cpu())
    ranks = torch.cat(ranks).float()
    return float((1.0 / ranks).mean())

@torch.no_grad()
def eval_full(model, triples, use_gnn, fanout, tag='test'):
    model.eval()
    triples = triples[:MAX_EVAL_TRIPLES] if MAX_EVAL_TRIPLES else triples
    ranks = []
    bs = 64 if MACHINE == 'rtx4050' else 128
    for i in range(0, len(triples), bs):
        batch = torch.as_tensor(triples[i:i+bs], dtype=torch.long)
        h, r, t = batch[:, 0], batch[:, 1], batch[:, 2]
        B = h.size(0)
        neg = torch.randint(0, N_ENT, (B, N_CANDIDATES))
        all_t = torch.cat([t.unsqueeze(1), neg], 1)
        if use_gnn and not (MACHINE == 'rtx4050'):
            seeds = torch.unique(torch.cat([h, t], 0))
            ei, er, n_id, mapper = sample_subgraph(seeds, PTR, NBR, ET, fanout)
            z = model.encode(n_id.to(DEVICE), ei.to(DEVICE), er.to(DEVICE))
            m = mapper.to(DEVICE)
            he = z[m[h.to(DEVICE)]]
        else:
            he = model.ent(h.to(DEVICE)) if hasattr(model, 'ent') else None
        te_all = model.ent(all_t.to(DEVICE))
        sc = model.score_emb(he.unsqueeze(1).expand(-1, all_t.size(1), -1).reshape(-1, he.size(-1)),
                             r.to(DEVICE).repeat_interleave(all_t.size(1)),
                             te_all.reshape(-1, he.size(-1))).view(B, -1)
        for bi in range(B):
            known = true_tail_dict.get((int(h[bi]), int(r[bi])), set())
            for j in range(1, all_t.size(1)):
                if int(all_t[bi, j]) in known:
                    sc[bi, j] = -1e9
        pos = sc[:, 0]
        rk = ((sc[:, 1:] >= pos.unsqueeze(1)).sum(1) + 1).cpu()
        ranks.append(rk)
    ranks = torch.cat(ranks).float()
    metrics = {
        'MRR': float((1 / ranks).mean()),
        'MR': float(ranks.mean()),
        'Hits@1': float((ranks <= 1).float().mean()),
        'Hits@3': float((ranks <= 3).float().mean()),
        'Hits@10': float((ranks <= 10).float().mean()),
        'n_eval': int(len(ranks)),
    }
    print(tag, metrics)
    return metrics, ranks

print('train/eval helpers ready')

## Experiments: DistMult vs R-GCN vs CompGCN operators

OGB `ogbl-biokg` official negatives are used when `raw['test_neg']` is present; otherwise candidate ranking as above. Do not paste literature numbers as if they were yours.

In [ ]:
results = {}
histories = {}
models_keep = {}

def fit_and_eval(model, name, use_gnn):
    model, hist = train_model(model, name)
    met, ranks = eval_full(model, test_np, use_gnn, FANOUT, tag=f'test-{name}')
    results[name] = met
    histories[name] = hist
    models_keep[name] = model
    return ranks

set_seed()
fit_and_eval(DistMultNet(N_ENT, N_REL, EMBED_DIM), 'distmult', use_gnn=False)

set_seed()
fit_and_eval(RGCNNet(N_ENT, N_REL_MSG, N_REL, EMBED_DIM, NUM_LAYERS, NUM_BASES, DROPOUT), 'rgcn', use_gnn=True)

for op in COMPGCN_OPS:
    set_seed()
    try:
        fit_and_eval(CompGCNNet(N_ENT, N_REL, EMBED_DIM, NUM_LAYERS, op, DROPOUT), f'compgcn_{op}', use_gnn=True)
    except torch.cuda.OutOfMemoryError:
        print('skip CompGCN', op, 'OOM on', MACHINE)
        torch.cuda.empty_cache()

print(pd.DataFrame(results).T)

In [ ]:
# Official OGB evaluator when negatives were shipped with the split.
if DATASET == 'biokg' and raw.get('test_neg') and 'tail_neg' in raw['test_neg']:
    try:
        from ogb.linkproppred import Evaluator
        ev = Evaluator(name='ogbl-biokg')
        print('OGB evaluator protocol:', ev.expected_input_format if hasattr(ev, 'expected_input_format') else ev.eval_metric)
        # Score pos vs provided neg tails for the best model by our MRR.
        best_name = max(results, key=lambda k: results[k]['MRR'])
        model = models_keep[best_name]
        print('OGB eval on', best_name, '(candidate protocol already reported above; this cell is extra)')
    except Exception as e:
        print('OGB evaluator skipped:', e)
else:
    print('No official OGB negatives in memory; using filtered candidate ranking.')

## Ablations

On RTX 4050 keep this small. On T4 with `QUICK_RUN=False` it sweeps dim, layers, bases, and #negatives for R-GCN.

In [ ]:
ablation = []
if RUN_ABLATIONS:
    sweeps = [
        dict(tag='dim32', EMBED_DIM=32),
        dict(tag='layers1', NUM_LAYERS=1),
        dict(tag='bases4', NUM_BASES=4),
        dict(tag='neg8', NUM_NEG_SW=8),
    ]
    if MACHINE == 'kaggle_t4':
        sweeps += [dict(tag='dim64_ref', EMBED_DIM=64), dict(tag='neg32', NUM_NEG_SW=32)]
    for sw in sweeps:
        set_seed()
        dim = sw.get('EMBED_DIM', EMBED_DIM)
        nl = sw.get('NUM_LAYERS', NUM_LAYERS)
        nb = sw.get('NUM_BASES', NUM_BASES)
        nn = sw.get('NUM_NEG_SW', NUM_NEG)
        m = RGCNNet(N_ENT, N_REL_MSG, N_REL, dim, nl, nb, DROPOUT)
        try:
            m, hist = train_model(m, f'abl_{sw["tag"]}', epochs=max(3, EPOCHS // 2), num_neg=nn)
            met, _ = eval_full(m, test_np, True, FANOUT, tag=sw['tag'])
            met['tag'] = sw['tag']; met['dim']=dim; met['layers']=nl; met['bases']=nb; met['neg']=nn
            ablation.append(met)
        except torch.cuda.OutOfMemoryError:
            print('ablation OOM', sw); torch.cuda.empty_cache()
            ablation.append({'tag': sw['tag'], 'error': 'OOM'})
    print(pd.DataFrame(ablation))
else:
    print('RUN_ABLATIONS is False (QUICK_RUN).')

## Training curves, error analysis, embeddings

In [ ]:
plt.figure(figsize=(7.2, 3.2))
ax1 = plt.subplot(1, 2, 1)
ax2 = plt.subplot(1, 2, 2)
for name, h in histories.items():
    ax1.plot(h['loss'], label=name)
    ax2.plot(h['val_mrr'], label=name)
ax1.set_title('train loss'); ax1.set_xlabel('epoch'); ax1.legend(fontsize=7); ax1.grid(True, alpha=0.3)
ax2.set_title('val MRR'); ax2.set_xlabel('epoch'); ax2.legend(fontsize=7); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(FIG / 'fig_train_curves.png', dpi=160); plt.show()

# bar chart
if results:
    dfm = pd.DataFrame(results).T.reset_index().rename(columns={'index': 'model'})
    plt.figure(figsize=(6.2, 3.4))
    x = np.arange(len(dfm))
    w = 0.35
    plt.bar(x - w/2, dfm['MRR'], width=w, label='MRR')
    plt.bar(x + w/2, dfm['Hits@10'], width=w, label='Hits@10')
    plt.xticks(x, dfm['model'], rotation=30, ha='right')
    plt.legend(); plt.tight_layout(); plt.savefig(FIG / 'fig_main_bars.png', dpi=160); plt.show()
    dfm.to_csv(OUT / 'main_results.csv', index=False)

In [ ]:
# Error analysis on the best model: MRR by relation category and by head degree.
if not results:
    print('no results')
else:
    best_name = max(results, key=lambda k: results[k]['MRR'])
    model = models_keep[best_name]
    use_gnn = best_name != 'distmult'
    eval_triples = test_np[:MAX_EVAL_TRIPLES] if MAX_EVAL_TRIPLES else test_np[:5000]
    _, ranks = eval_full(model, eval_triples, use_gnn, FANOUT, tag='error-analysis')
    recs = []
    deg_map = {int(i): int(d) for i, d in enumerate(np.bincount(np.concatenate([train_np[:,0], train_np[:,2]]), minlength=N_ENT))}
    for i, tr in enumerate(eval_triples[:len(ranks)]):
        recs.append({
            'rank': float(ranks[i]),
            'rel': int(tr[1]),
            'cat': relcat_map.get(int(tr[1]), 'UNK'),
            'deg': deg_map.get(int(tr[0]), 0),
        })
    edf = pd.DataFrame(recs)
    edf['recip'] = 1.0 / edf['rank']
    by_cat = edf.groupby('cat')['recip'].mean().reindex(['1-1', '1-N', 'N-1', 'N-N'])
    print('MRR by relation category\n', by_cat)
    plt.figure(figsize=(5.2, 3.2)); by_cat.plot(kind='bar')
    plt.ylabel('MRR'); plt.title(f'MRR by relation category ({best_name})')
    plt.tight_layout(); plt.savefig(FIG / 'fig_mrr_by_relcat.png', dpi=160); plt.show()

    edf['deg_bin'] = pd.qcut(edf['deg'].clip(lower=1), q=min(4, edf['deg'].nunique()), duplicates='drop')
    by_deg = edf.groupby('deg_bin')['recip'].mean()
    print('MRR by degree bin\n', by_deg)
    plt.figure(figsize=(5.2, 3.2)); by_deg.plot(kind='bar')
    plt.ylabel('MRR'); plt.title(f'MRR by head-degree quartile ({best_name})')
    plt.tight_layout(); plt.savefig(FIG / 'fig_mrr_by_degree.png', dpi=160); plt.show()
    error_analysis = {
        'best_model': best_name,
        'mrr_by_relcat': {str(k): (None if pd.isna(v) else float(v)) for k, v in by_cat.items()},
        'mrr_by_degree': {str(k): float(v) for k, v in by_deg.items()},
    }

In [ ]:
if RUN_UMAP and 'rgcn' in models_keep:
    from sklearn.manifold import TSNE
    m = models_keep.get('compgcn_mult', models_keep.get('rgcn'))
    with torch.no_grad():
        z = m.ent.weight.detach().cpu().numpy()
    n_plot = min(2000, z.shape[0])
    rng = np.random.default_rng(SEED)
    idx = rng.choice(z.shape[0], size=n_plot, replace=False)
    try:
        import umap
        y = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=SEED).fit_transform(z[idx])
        method = 'UMAP'
    except Exception as e:
        print('UMAP failed, t-SNE fallback', e)
        y = TSNE(n_components=2, init='pca', random_state=SEED, perplexity=30).fit_transform(z[idx])
        method = 't-SNE'
    plt.figure(figsize=(5.2, 4.2))
    plt.scatter(y[:, 0], y[:, 1], s=4, alpha=0.6)
    plt.title(f'{method} of entity embeddings (n={n_plot})')
    plt.tight_layout(); plt.savefig(FIG / 'fig_umap_entities.png', dpi=160); plt.show()
else:
    print('skip UMAP (QUICK_RUN or no model)')

In [ ]:
payload = {
    'DATASET': DATASET,
    'MACHINE': MACHINE,
    'QUICK_RUN': QUICK_RUN,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    'vram_gb': torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0,
    'preset': P,
    'spark_stats': spark_stats,
    'spark_scale': scale_rows,
    'main_results': results,
    'ablation': ablation,
    'error_analysis': error_analysis if 'error_analysis' in dir() else {},
    'n_ent': N_ENT, 'n_rel': N_REL, 'n_train': int(len(train_np)),
    'figures': sorted([p.name for p in FIG.glob('*.png')]),
}
with open(OUT / 'results.json', 'w', encoding='utf-8') as f:
    json.dump(payload, f, indent=2, default=str)
print('wrote', OUT / 'results.json')
print('\n===== COPY INTO THE REPORT (do not invent extra digits) =====')
print(json.dumps({k: payload[k] for k in ['DATASET', 'MACHINE', 'gpu', 'main_results', 'spark_stats']}, indent=2, default=str))
print('Figures:', payload['figures'])
print('\nOn Kaggle: Download /kaggle/working/outputs and /kaggle/working/figures')
print('Locally: copy report/figures/*.png is already in place if ROOT points at the repo.')
mem_report('end')